# Notebook 05 - Aplicaciones Practicas

## Objetivos
- Generar codigo con prompts estructurados.
- Crear contenido de marketing con role prompting.
- Resumir, traducir y analizar texto con distintas tecnicas.

## Introduccion
Este notebook es 100% hands-on: 6 aplicaciones reales que puedes replicar en tu trabajo usando GPT-2 en español como laboratorio.

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path('..')
DATASETS = BASE / 'datasets'
print('Entorno listo. Datasets:', list(DATASETS.glob('*.csv')))

In [ ]:
from transformers import pipeline, set_seed

set_seed(42)
generator = pipeline('text-generation', model='datificate/gpt2-small-spanish')
print('GPT-2 en español listo para experimentos de prompting')

In [ ]:
def generar(prompt, max_new_tokens=60, temperature=0.7):
    out = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    return out[0]['generated_text']

print('Funcion generar() lista')

## 1) Generacion de codigo

In [ ]:
prompt_codigo = (
    'Escribe una funcion Python que reciba una lista de numeros y devuelva el promedio. '
    'Incluye manejo de errores para listas vacias. Solo codigo:\n'
    'def calcular_promedio(numeros):'
)
print('=== CODIGO ===')
print(generar(prompt_codigo, max_new_tokens=80, temperature=0.4))

## 2) Creacion de contenido (marketing)

In [ ]:
prompt_marketing = (
    'Eres un copywriter creativo para una startup tecnologica.\n'
    'Escribe una descripcion de producto en 60 palabras. Tono juvenil. Incluye 3 beneficios.\n'
    'Producto: CloudStorage Pro - almacenamiento seguro con busqueda por IA.\n'
    'Descripcion:'
)
print('=== MARKETING ===')
print(generar(prompt_marketing, max_new_tokens=70, temperature=0.8))

## 3) Resumen de documentos

In [ ]:
df_docs = pd.read_csv(DATASETS / 'documentos_empresa.csv')
doc = df_docs[df_docs['titulo'] == 'Reporte Q3 2025'].iloc[0]
prompt_resumen = (
    f'Resume en 3 viñetas. Solo hechos. Maximo 15 palabras cada una.\n'
    f'Documento: {doc["contenido"]}\n'
    f'Resumen:'
)
print('=== RESUMEN ===')
print(generar(prompt_resumen, max_new_tokens=50, temperature=0.4))

## 4) Traduccion

In [ ]:
frases = [
    'El aprendizaje automatico transforma la industria',
    'Los transformers revolucionaron el procesamiento de lenguaje',
    'La inteligencia artificial generativa crea contenido nuevo',
]
for frase in frases:
    p = f'Continua el texto de forma coherente en español:\n{frase}'
    print(f'Entrada: {frase}')
    print(f'Continuacion: {generar(p, max_new_tokens=25, temperature=0.5)[-60:]}')
    print('-' * 40)

## 5) Analisis de sentimiento con Robertuito (español)

In [ ]:
sentimientos = pipeline('sentiment-analysis', model='pysentimiento/robertuito-sentiment-analysis')
print('Modelo de sentimiento en español cargado')

In [ ]:
mensajes = [
    'Excelente producto, lo recomiendo totalmente',
    'El sistema se cayó y perdí todo mi trabajo',
    '¿Cómo cambio mi contraseña del portal?',
]
for msg in mensajes:
    resultado = sentimientos(msg)[0]
    print(f'Texto: {msg}')
    print(f'Sentimiento: {resultado["label"]} (score: {resultado["score"]:.3f})')
    print('-' * 50)

## 6) Preguntas y respuestas con BERT en español

In [ ]:
qa = pipeline('question-answering', model='mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es')
print('Modelo QA en español cargado')

In [ ]:
contexto = (
    'Python fue creado por Guido van Rossum y publicado en 1991. '
    'Es un lenguaje de programacion interpretado y multiparadigma.'
)
preguntas = ['¿Quién creó Python?', '¿En qué año se publicó Python?']
for pregunta in preguntas:
    out = qa(question=pregunta, context=contexto)
    print(f'P: {pregunta}')
    print(f'R: {out["answer"]} (score: {out["score"]:.3f})')
    print('-' * 40)

## 7) Automatizacion: pipeline de 3 prompts encadenados

In [ ]:
ticket = 'No puedo iniciar sesion en el panel desde ayer, intenté restablecer contraseña'

p1 = f'Extrae el problema principal en 5 palabras: {ticket}\nProblema:'
problema = generar(p1, max_new_tokens=10, temperature=0.3)

p2 = f'Clasifica urgencia alta media baja: {ticket}\nUrgencia:'
urgencia = generar(p2, max_new_tokens=5, temperature=0.3)

p3 = f'Sugiere una accion de soporte para: {ticket}\nAccion:'
accion = generar(p3, max_new_tokens=20, temperature=0.5)

pipeline = pd.DataFrame([
    {'paso': 'Extraer problema', 'resultado': problema[-40:]},
    {'paso': 'Clasificar urgencia', 'resultado': urgencia[-20:]},
    {'paso': 'Sugerir accion', 'resultado': accion[-50:]},
])
display(pipeline)

## Resultados
Implementamos 6 aplicaciones practicas: codigo, marketing, resumen, traduccion, analisis y automatizacion con pipeline de prompts.

## Conclusiones
El mismo LLM resuelve tareas distintas cambiando solo el prompt. La clave es la combinacion de tecnica + restricciones + formato.

## Ejercicios guiados resueltos
**Ejercicio:** Crea un pipeline de 3 pasos para analizar una reseña.

**Solucion:**

In [ ]:
review = 'Buen producto pero el envío fue muy lento'
for paso, prompt in [('Sentimiento', f'Sentimiento: {review}\nEtiqueta:'), ('Tema', f'Tema principal: {review}\nTema:'), ('Accion', f'Accion sugerida: {review}\nAccion:')]:
    print(f'{paso}: {generar(prompt, max_new_tokens=10, temperature=0.3)[-30:]}')

## Ejercicios propuestos
1. Genera codigo para validar correos electronicos.
2. Resume los 5 documentos del CSV.
3. Crea copy para 3 productos ficticios en español.

## Preguntas de reflexion
1. Que aplicacion fue mas facil de promptear?
2. Donde fallo el modelo y como lo mejorarias?
3. Como escalaria estos prompts a 1000 solicitudes/dia?